# Revision: Reading Messy JSON

This notebook covers working with real, nested Twitter-data-shaped JSON (the format produced by `twarc`, the tool used earlier in the course to collect Twitter data), plus the conceptual "why" questions several of you raised in the Week 3 discussion (why merge is required, why pseudonymize, how to make open-ended decisions, how to judge sentiment disagreements) and a couple of practical aspect.

## Setup

In [3]:
import pandas as pd
import re
import json

# How is JSON different from a DataFrame?

Before opening the real file, let's build up the idea with a tiny made-up example.

**JSON** is a way of writing data as plain *text*, so it can be sent from one computer program to another (for example, from Twitter's servers to your laptop). Since you are familiar with Python dictionary before, JSON will look familiar: it uses the same `{key: value}` style, just written out as one long piece of text instead of being a "live" Python object in memory yet.

Here's a tiny JSON example, written as a Python string so you can see what the raw text looks like:


In [18]:
json_text = '{"id": "123", "text": "Hello world!", "public_metrics": {"like_count": 5, "retweet_count": 1}}'
print(json_text)
print(type(json_text))

{"id": "123", "text": "Hello world!", "public_metrics": {"like_count": 5, "retweet_count": 1}}
<class 'str'>


Right now, `json_text` is just one long string. Python doesn't yet know it *means* a dictionary with an `id`, a `text`, and so on — you can't write `json_text['text']`, because to Python it's still just text, like a sentence in an email.

The `json` module's `.loads()` function ("load string") is what turns that text into an actual, usable Python dictionary:


In [5]:
tweet_dict = json.loads(json_text)
print(tweet_dict)
print(type(tweet_dict))
tweet_dict['text']

{'id': '123', 'text': 'Hello world!', 'public_metrics': {'like_count': 5, 'retweet_count': 1}}
<class 'dict'>


'Hello world!'

Notice that `public_metrics` is a dictionary sitting *inside* the outer dictionary. This is the key difference to keep in mind for the rest of this notebook:

* A **DataFrame** is a flat table: every row has the same columns, and each cell normally holds one simple value (a number, a bit of text, a date).
* **JSON** doesn't have to be flat at all: one field can hold a whole list, or another dictionary, or a list of dictionaries — nested as many levels deep as needed.

That's exactly why you can't just hand a JSON file to pandas and expect a clean, ready-to-use table back the way `pd.read_csv()` gives you one. Pandas needs *you* (or a helper function you write) to first decide how to unpack those nested pieces into plain columns. That unpacking process is often called **"flattening,"** and it's what most of this notebook is about.

### What about JSONL?

The file you'll use in this notebook isn't a single JSON object — Twitter sends results back in batches ("pages"), so a real data collection run produces not one tweet but hundreds, gathered across many separate pages. `twarc` saves each page as its own JSON object, one per line, in a file. That's what **JSONL** ("JSON Lines") means: **many separate JSON objects, one per line** — not one giant JSON list containing everything.

Here's a tiny 2-line JSONL example (as a string, with `\n` marking where a new line starts):


In [6]:
jsonl_text = '{"id": "1", "text": "First tweet"}\n{"id": "2", "text": "Second tweet"}'
print(jsonl_text)

{"id": "1", "text": "First tweet"}
{"id": "2", "text": "Second tweet"}


Notice there's no `,` joining the two objects and no `[ ]` wrapping them the way a Python list would have — they really are two separate, independent JSON documents that just happen to live in the same file, one per line. To read this properly, you have to split it into lines first, then parse each line **on its own**:


In [7]:
lines = jsonl_text.splitlines()          # split the text into a list of lines
parsed = [json.loads(line) for line in lines]   # turn each line into its own dictionary
parsed

[{'id': '1', 'text': 'First tweet'}, {'id': '2', 'text': 'Second tweet'}]

That loop is the manual, do-it-yourself version of what `pd.read_json(..., lines=True)` does for you automatically: it reads the file one line at a time, and parses each line as its own separate JSON object. Now that you've seen *why* that's necessary, the next question — and the error it deliberately walks you into — will make a lot more sense.


# Part 1 — Reading nested JSON

Now for the real file. This is the part of Week 3 that tripped up the most people: the data doesn't arrive as tidy rows and columns, it arrives as **nested** JSON (as you just saw in Part 0), and you need to build your own tidy dataframe out of it before any of the pandas skills from Week 2 are usable.


## Question 1 — Load the file (the wrong way first, on purpose)

Try loading `synthetic_tweets_raw.jsonl` with `pd.read_json('da3_data/synthetic_tweets_raw.jsonl')` — no extra arguments. Read the error message. Then fix it by adding `lines=True`.

**Why this matters:** The `Trailing Data` (or `ValueError`) error is one of the most common blockers with real Twitter/API data, and it looks scary but has one specific cause. A `.jsonl` file is **not** one big JSON document — it's many separate JSON objects, one per line (that's what the 'L' means). `pd.read_json()` assumes a single document by default, so it parses the first line successfully, then finds more data after it and gives up. `lines=True` tells it to parse the file line-by-line instead. Once you know this, the error stops being scary and becomes a one-argument fix.

**Hint:**
```python
df_jsonl = pd.read_json('da3_data/synthetic_tweets_raw.jsonl')  # will error
df_jsonl = pd.read_json('da3_data/synthetic_tweets_raw.jsonl', lines=True)  # fixed
```

In [8]:
# ANSWER (delete before distributing)
try:
    df_jsonl = pd.read_json('da3_data/synthetic_tweets_raw.jsonl')
except ValueError as e:
    print('Error:', e)

df_jsonl = pd.read_json('da3_data/synthetic_tweets_raw.jsonl', lines=True)
df_jsonl.shape

Error: Trailing data


(14, 3)

## Question 2 — Inspect the raw (still nested) structure

Look at `df_jsonl.columns`, `df_jsonl.dtypes`, and then `df_jsonl['data'].iloc[0]` — the first row's `data` value. What Python type is it? What's inside it?

**Why this matters:** This is the step people skip, and it's the one that actually builds understanding. `df_jsonl` loaded successfully, but it is **not** a tidy tweet-per-row table yet — each row is one API 'page,' and the actual tweets are buried inside a Python list (inside the `data` column) of dictionaries. Nothing in `df_jsonl` is directly usable for analysis (you can't `.groupby()` a column full of lists). Seeing this with your own eyes — instead of just running someone else's flattening code — is what makes the *next* question make sense instead of feeling like magic.

**Hint:**
```python
df_jsonl.dtypes
type(df_jsonl['data'].iloc[0])
df_jsonl['data'].iloc[0][0]  # the first tweet dict on the first page
```

In [9]:
# ANSWER (delete before distributing)
print(df_jsonl.dtypes)
print(type(df_jsonl['data'].iloc[0]))
print(len(df_jsonl['data'].iloc[0]), 'tweets on the first page')
df_jsonl['data'].iloc[0][0]

data        object
includes    object
meta        object
dtype: object
<class 'list'>
25 tweets on the first page


{'id': '1800000000000000000',
 'author_id': '910000000000001211',
 'created_at': '2024-08-28T23:44:00.000Z',
 'text': 'Just read a great thread on carbon emissions, worth your time. cc @greenwatch8285_42',
 'lang': 'es',
 'public_metrics': {'retweet_count': 3,
  'reply_count': 3,
  'like_count': 14,
  'quote_count': 3}}

## Question 3 — Flatten the tweets into one row per tweet

Write a function `get_tweets(df)` that loops over every page's `data` list and stacks all the individual tweet dictionaries into a single flat dataframe (one row per tweet). Apply it to get a `tweets` dataframe. Confirm `len(tweets)` matches the total tweet count you'd get by manually summing the length of `data` across all pages.

**Why this matters:** This is exactly the code Qi Ruan described running without fully understanding. Walking through it line-by-line: `pd.DataFrame(item)` turns *one page's* list-of-dicts into a small dataframe; looping over every page and `pd.concat`-ing those small dataframes together builds the full table. Once you've written it yourself (not just executed it), 'why is it written that way' stops being a mystery — it's just 'flatten each page, then stack the pages.'

**Hint:**
```python
def get_tweets(df):
    results = pd.DataFrame()
    for item in df['data'].values.tolist():
        results = pd.concat([results, pd.DataFrame(item)])
    return results.reset_index(drop=True)

tweets = get_tweets(df_jsonl)
len(tweets)
```

In [10]:
# ANSWER (delete before distributing)
def get_tweets(df):
    results = pd.DataFrame()
    for item in df['data'].values.tolist():   # item = one page's list of tweet dicts
        results = pd.concat([results, pd.DataFrame(item)])  # turn that list into rows, stack onto the rest
    return results.reset_index(drop=True)

tweets = get_tweets(df_jsonl)

manual_count = sum(len(page) for page in df_jsonl['data'])
print(len(tweets), manual_count, len(tweets) == manual_count)
tweets.head()

352 352 True


,id,author_id,created_at,text,lang,public_metrics
0,1800000000000000000,910000000000001211,2024-08-28T23:44:00.000Z,"Just read a great thread on carbon emissions, ...",es,"{'retweet_count': 3, 'reply_count': 3, 'like_c..."
1,1800000000000000097,910000000000006055,2024-08-25T14:35:00.000Z,Local government just announced new funding fo...,en,"{'retweet_count': 3, 'reply_count': 3, 'like_c..."
2,1800000000000000194,910000000000002941,2024-08-17T02:54:00.000Z,RT @ecohub3135_30: My research this year has f...,en,"{'retweet_count': 8, 'reply_count': 2, 'like_c..."
3,1800000000000000291,910000000000003114,2024-09-11T06:04:00.000Z,Proud to see my city investing in green techno...,en,"{'retweet_count': 4, 'reply_count': 4, 'like_c..."
4,1800000000000000388,910000000000001211,2024-09-13T15:18:00.000Z,RT @stormaction6537_13: Proud to see my city i...,es,"{'retweet_count': 3, 'reply_count': 3, 'like_c..."


## Question 4 — Expand the nested `public_metrics` column

Look at `tweets['public_metrics'].iloc[0]` — it's still a dictionary (nested one level deeper than `data` was). Write a function that expands a row's `public_metrics` dict into separate columns (e.g. `metric_retweet_count`), apply it to every row, and confirm the new columns exist.

**Why this matters:** Twitter's API (and many others) nests *related* numbers together instead of flattening them for you — `public_metrics` groups all the engagement counts into one dictionary per tweet. That's convenient for whoever designed the API, but not for your analysis: you can't plot or filter a column full of dictionaries. This is the second (and last) layer of nesting in this dataset — after this step, `tweets` is a genuinely flat, analysis-ready dataframe.

**Hint:**
```python
def expand_metrics(row):
    if isinstance(row['public_metrics'], dict):  # isinstance checks: is this value still a dictionary?
        for key, value in row['public_metrics'].items():
            row['metric_' + key] = value
    return row

tweets = tweets.apply(expand_metrics, axis=1)
```

In [11]:
# ANSWER (delete before distributing)
def expand_metrics(row):
    if isinstance(row['public_metrics'], dict):  # isinstance checks: is this value still a dictionary?
        for key, value in row['public_metrics'].items():
            row['metric_' + key] = value
    return row

tweets = tweets.apply(expand_metrics, axis=1)
tweets[['metric_retweet_count', 'metric_reply_count', 'metric_like_count', 'metric_quote_count']].head()

,metric_retweet_count,metric_reply_count,metric_like_count,metric_quote_count
0,3.0,3.0,14.0,3.0
1,3.0,3.0,10.0,1.0
2,8.0,2.0,24.0,10.0
3,4.0,4.0,21.0,1.0
4,3.0,3.0,26.0,22.0


## Question 5 — Flatten the users the same way

Do the same two-step flattening (stack the pages, then expand `public_metrics`) for the *users*, which live in `df_jsonl['includes']`, under the key `'users'`. Call the result `users_raw`. Then check: is `len(users_raw)` bigger than `len(tweets)`? Why?

**Why this matters:** Same nesting problem, one level differently shaped: `includes` is a dict per page (not a list), and the list you actually want is inside it at `includes['users']`. If this feels repetitive — it is. That's deliberate: once you can flatten *any* nested list-of-dicts, you can handle a `data` field, an `includes.users` field, or a completely different nested API's payload the same way. This `users_raw` table is exactly what Week 3's minimization/pseudonymization work (in `wtt_w3_inclass.ipynb`) starts from.

**Hint:**
```python
def get_users(df):
    results = pd.DataFrame()
    for item in df['includes'].values.tolist():
        results = pd.concat([results, pd.DataFrame(item['users'])])
    return results.reset_index(drop=True)

users_raw = get_users(df_jsonl)
users_raw = users_raw.apply(expand_metrics, axis=1)
```

In [12]:
# ANSWER (delete before distributing)
def get_users(df):
    results = pd.DataFrame()
    for item in df['includes'].values.tolist():   # item = one page's includes dict
        results = pd.concat([results, pd.DataFrame(item['users'])])  # the list is nested one key deeper
    return results.reset_index(drop=True)

users_raw = get_users(df_jsonl)
users_raw = users_raw.apply(expand_metrics, axis=1)

print(len(tweets), len(users_raw))
# users_raw is bigger: a user appears once per tweet they authored AND once per tweet
# where they were mentioned - the same duplication pattern from wtt_w3_inclass, just
# now you've seen exactly where it comes from in the raw API structure.

352 579


## Question 6 — Know when to go back to the raw JSON

Pick one tweet id from `tweets` and manually find that *same* tweet inside the original nested `df_jsonl['data']` (without using your `get_tweets` function) by looping through pages and checking each tweet dict's `'id'`. Confirm the text matches.

**Why this matters:** Once flattening functions exist, it's tempting to forget the raw JSON entirely and only ever look at the flat dataframe. But when something in your flat table looks wrong (a value you didn't expect, a field that seems missing), the raw JSON is still your source of truth — going back to it lets you check whether *your flattening code* has a bug, or whether the *original data* actually looks like that. This is a debugging habit, not just a one-off exercise.

**Hint:**
```python
target_id = tweets['id'].iloc[0]
for page in df_jsonl['data']:
    for tw in page:
        if tw['id'] == target_id:
            print(tw)
```

In [13]:
# ANSWER (delete before distributing)
target_id = tweets['id'].iloc[0]

found = None
for page in df_jsonl['data']:
    for tw in page:
        if tw['id'] == target_id:
            found = tw
            break
    if found:
        break

print(found['text'] == tweets[tweets['id'] == target_id]['text'].iloc[0])
found

True


{'id': '1800000000000000000',
 'author_id': '910000000000001211',
 'created_at': '2024-08-28T23:44:00.000Z',
 'text': 'Just read a great thread on carbon emissions, worth your time. cc @greenwatch8285_42',
 'lang': 'es',
 'public_metrics': {'retweet_count': 3,
  'reply_count': 3,
  'like_count': 14,
  'quote_count': 3}}

# Part 2 — The "why", not just the "how"

A few of you said you could follow the code but wanted to understand *why* a step was needed at all. These are short explanatory notes, each paired with a small check in code so you can see the idea in action rather than just read about it.


## Question 7 — Why does merge() even exist here?

Read the note below, then merge your flattened `tweets` and a deduplicated `users_raw` (drop duplicates on `id` first) so each tweet has its author's follower count attached. Confirm the new column is there.

**Why this matters:** Twitter's API doesn't repeat a user's full profile information on every single tweet they post — that would mean sending (and storing) the same name, bio, and follower count over and over again if one popular account posted 50 times in your dataset. Instead, the API sends tweets and users as two **separate** lists (exactly what you just built: `tweets` and `users_raw`) and leaves it to *you* to reconnect them when you need to. `merge()` is that reconnection step. It's not an arbitrary pandas trick you have to memorize — it's the direct, logical consequence of how the data was split up to begin with.

**Hint:**
```python
users_unique = users_raw.drop_duplicates(subset=['id'])
tweets_with_author = tweets.merge(users_unique, how='left', left_on='author_id', right_on='id', suffixes=('_tweet', '_user'))
```

In [14]:
# ANSWER (delete before distributing)
users_unique = users_raw.drop_duplicates(subset=['id'])

tweets_with_author = tweets.merge(users_unique, how='left', left_on='author_id', right_on='id',
                                   suffixes=('_tweet', '_user'))
'metric_followers_count_user' in tweets_with_author.columns or 'metric_followers_count' in tweets_with_author.columns

True

## Question 8 — Why removing `username` isn't enough on its own

Write a function `find_leaked_mentions(text)` that checks whether a piece of text still contains an `@username`-style mention. Apply it to `tweets['text']` and count how many tweets would still reveal a specific person's handle even *after* you've deleted the `username`/`id` columns from your minimized dataframe.

**Why this matters:** One of you pointed this out, and it's a genuinely sharp observation: privacy isn't just about which *columns* you keep. `del df_min['username']` removes the column — but if the free-text `text` column still contains `@someone_else`, that person's handle is still sitting in your "anonymized" dataset. Structured columns (like `username`) are easy to reason about; unstructured text is where identifying information can hide in plain sight. This is exactly why `wtt_w3_inclass.ipynb`'s Question 15 replaces `@mentions` inside the text itself, separately from removing the `username` column.

**Hint:**
```python
def find_leaked_mentions(text):
    return bool(re.search(r'@\w+', text))

tweets['text'].apply(find_leaked_mentions).sum()
```

In [15]:
# ANSWER (delete before distributing)
def find_leaked_mentions(text):
    return bool(re.search(r'@\w+', text))

leaked = tweets['text'].apply(find_leaked_mentions)
print(f'{leaked.sum()} of {len(tweets)} tweets still contain an @mention in the text')

204 of 352 tweets still contain an @mention in the text


## Question 9 — There often isn't one 'correct' minimized dataset

Below are two different research questions. For each, write the list of columns from `tweets_with_author` you'd keep, with a one-line comment justifying each column.

- RQ-A: *Does the sentiment of a tweet's text predict how many replies it gets?*
- RQ-B: *Are verified accounts more likely to post about certain topics than unverified accounts?*

**Why this matters:** Several of you said this felt uncomfortably open-ended, like you might be doing it 'wrong.' You're not missing a rule — there generally isn't a single correct minimized dataset, because the right columns depend entirely on the RQ. RQ-A needs `text` and `reply_count` at minimum, plus anything you'd control for (e.g. follower count, since popular accounts get more replies regardless of sentiment). RQ-B doesn't need `text` at all, but does need `verified` and a topic label. Seeing two different RQs produce two different 'correct' answers from the *same* raw data is the point — it should make the open-endedness feel normal rather than like something you're getting wrong.

**Hint:**
```python
cols_for_rq_a = ['id', 'text', 'metric_reply_count', ...]  # + control variables
cols_for_rq_b = ['id', 'verified', ...]  # topic label, no text needed
```

In [16]:
# ANSWER (delete before distributing)
cols_for_rq_a = [
    'id',                        # tweet identifier
    'text',                      # to compute/inspect sentiment
    'metric_reply_count',        # outcome variable
    'metric_followers_count',    # control: popular accounts get more replies regardless of sentiment
]

cols_for_rq_b = [
    'id',                        # tweet identifier
    'verified',                  # the grouping variable of interest
    # a topic/category column would go here once created (e.g. Question 6 in wtt_w3_inclass)
]

print(cols_for_rq_a)
print(cols_for_rq_b)

['id', 'text', 'metric_reply_count', 'metric_followers_count']
['id', 'verified']


## Question 10 — Judging a disagreement with a sentiment score

Take 5 tweets from `tweets`. For each, write down your *own* one-word sentiment judgement (positive/negative/neutral) as a small dictionary or list, next to the text. (No sentiment model needed here — this question is about the judging process itself, which you'll apply to real model output in `wtt_w3_inclass.ipynb`.)

**Why this matters:** A couple of you asked, reasonably: if I disagree with the algorithm's score, is that a mistake or a valid different reading? There's no universal rule, but here's a useful way to think about it: sentiment tools score text **mechanically** — mostly by recognizing individual words as positive or negative, with a few simple adjustments for things like the word 'not', punctuation, or ALL CAPS. They don't understand sarcasm, context, or the specific topic the way you do. A disagreement on a genuinely ambiguous or sarcastic tweet is expected and worth mentioning in your write-up, not something to 'fix.' A disagreement on a tweet that's clearly positive or negative to any human reader is more likely a real limitation of the tool (or a bug in how you built your combined score) — worth double-checking.

**Hint:**
```python
sample = tweets.sample(5, random_state=1)[['id', 'text']]
my_labels = {row.id: 'positive'  # ... fill in your own reading for each
             for row in sample.itertuples()}
```

In [17]:
# ANSWER (delete before distributing)
sample = tweets.sample(5, random_state=1)[['id', 'text']]
print(sample.to_string())

# fill these in by reading the 5 texts above
my_labels = {
    # tweet_id: 'positive' / 'negative' / 'neutral'
}

                      id                                                                                                                          text
150  1800000000000014550                              Can we talk about how climate policy affects everyday people? #Sustainability cc @solarhub9552_1
169  1800000000000016393                                                      This graph on carbon emissions is wild, take a look. cc @bluedesk3458_26
91   1800000000000008827                          Can we talk about how wildlife conservation affects everyday people? #Environment cc @ecodaily3518_0
299  1800000000000029003                                               My research this year has focused on climate activism, happy to share findings.
298  1800000000000028906  RT @solardaily1029_3: @futurereport3198_19 This graph on the youth climate movement is wild, take a look. cc @gridnow8990_25


## Question 11 — Fix the scrambled pipeline

The 5 lines below are the correct Week 3 pipeline, but shuffled — running them in this order raises a `NameError` (this is exactly the error some of you hit when `df_min` wasn't defined yet). Reorder them into a version that runs top to bottom without error.

```python
del df_min['username']
tweets_flat = get_tweets(df_jsonl)
df_min = merged.merge(users_unique, how='left', left_on='author_id', right_on='id')[['username', 'text']]
users_unique = get_users(df_jsonl).drop_duplicates(subset=['id'])
merged = tweets_flat.copy()
```

**Why this matters:** This is the single most common non-conceptual bug in this kind of notebook: a cell fails with `NameError: name 'X' is not defined` not because the *logic* is wrong, but because a cell defining `X` further up either wasn't run yet, or was deleted. Before debugging the logic of a failing cell, always ask first: 'has every cell above this one actually been executed, in order, in this kernel?' Restart & Run All is the fastest way to check.

**Hint:**
```python
# think about what each line needs to already exist before it can run,
# then order them so every variable is defined before it's used
```

In [ ]:
# ANSWER (delete before distributing)
# correct order:
tweets_flat = get_tweets(df_jsonl)
users_unique = get_users(df_jsonl).drop_duplicates(subset=['id'])
merged = tweets_flat.copy()
df_min = merged.merge(users_unique, how='left', left_on='author_id', right_on='id')[['username', 'text']]
del df_min['username']

df_min.head()

## Question 12 — Write a merge sanity-check

Write a function `check_merge(left_df, right_df, result_df, how)` that uses Python's `assert` keyword to stop with a clear error message if the merged result's length doesn't match what's expected for that `how` (e.g. for `how='left'`, the result should never end up with *fewer* rows than the left dataframe). Use it to check your Question 7 merge.

**Why this matters:** This directly answers 'how do I know if the row count I got is normal or means something went wrong.' `assert some_condition, 'error message'` is a simple Python tool: if the condition is `True`, nothing happens and your code keeps running; if it's `False`, Python immediately stops and shows your message (raising an `AssertionError`). That turns 'does this look right?' from a guess into an automatic yes/no check — a small habit worth keeping and reusing across every merge you do for the rest of the course, not just this one.

**Hint:**
```python
def check_merge(left_df, right_df, result_df, how):
    if how == 'left':
        assert len(result_df) == len(left_df), f'expected {len(left_df)} rows, got {len(result_df)}'
    # ... add the same idea for 'right' and 'inner'
```

In [ ]:
# ANSWER (delete before distributing)
def check_merge(left_df, right_df, result_df, how, on_left=None, on_right=None):
    if how == 'left':
        assert len(result_df) >= len(left_df), 'left merge should never drop left rows'
    elif how == 'right':
        assert len(result_df) >= len(right_df), 'right merge should never drop right rows'
    elif how == 'inner':
        assert len(result_df) <= min(len(left_df), len(right_df)), 'inner merge cannot exceed the smaller table'
    elif how == 'outer':
        assert len(result_df) >= max(len(left_df), len(right_df)), 'outer merge cannot be smaller than either table'
    print(f'OK: {how} merge produced {len(result_df)} rows (left={len(left_df)}, right={len(right_df)})')

check_merge(tweets, users_unique, tweets_with_author, how='left')